In [1]:
"""
Cricsheet IPL 2026 Data Loader
===============================
Loads IPL 2026 season data from a full Cricsheet IPL archive folder
and produces 4 clean dataframes/CSV files:
 
    players.csv    — one row per player (name + Cricsheet ID)
    matches.csv    — one row per match (venue, teams, result etc.)
    deliveries.csv — one row per ball bowled
    officials.csv - one row per official
 
Only matches with ID >= 1527674 are loaded (start of 2026 IPL season).
 
Usage:
    1. Set DATA_FOLDER to the path of your folder
    2. Run: python cricket_loader.py
"""

'\nCricsheet IPL 2026 Data Loader\n===============================\nLoads IPL 2026 season data from a full Cricsheet IPL archive folder\nand produces 4 clean dataframes/CSV files:\n\n    players.csv    — one row per player (name + Cricsheet ID)\n    matches.csv    — one row per match (venue, teams, result etc.)\n    deliveries.csv — one row per ball bowled\n    officials.csv - one row per official\n\nOnly matches with ID >= 1527674 are loaded (start of 2026 IPL season).\n\nUsage:\n    1. Set DATA_FOLDER to the path of your folder\n    2. Run: python cricket_loader.py\n'

### Import Python libraries and initialize constants

In [2]:
import pandas as pd
import os
import json 
import numpy as np 

IPL_DATA_FOLDER = "./ipl_male_json"
FIRST_2026_REG_SEASON_MATCH = 1527674
LAST_2026_REG_SEASON_MATCH = 1529313
FINAL_MATCH = 1535465
#pd.set_option('display.max_rows', None)

### Get 2026 Match IDs

In [3]:
def get_2026_match_IDs(folder_path):
    """
    Scans the folder for all delivery CSVs and returns only
    match IDs >= FIRST_2026_MATCH.
    """
    all_ids = []
    all_reg_season_ids = []
    all_playoff_ids = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            try:
                match_id = int(filename.replace(".json", ""))
                if match_id >= FIRST_2026_REG_SEASON_MATCH:
                    all_ids.append(match_id)
            except ValueError:
                pass  # skip any files that don't have a numeric name
 
    all_ids.sort()
    all_playoff_ids = all_ids[-4:]
    all_reg_season_ids = all_ids[:-4]
    return all_ids, all_reg_season_ids, all_playoff_ids

In [4]:
match_ids, reg_season_ids, playoff_ids = get_2026_match_IDs(IPL_DATA_FOLDER)
match_ids

[1527674,
 1527675,
 1527676,
 1527677,
 1527678,
 1527679,
 1527680,
 1527681,
 1527682,
 1527683,
 1527684,
 1527685,
 1527686,
 1527687,
 1527688,
 1527689,
 1527690,
 1527691,
 1527692,
 1527693,
 1529264,
 1529265,
 1529266,
 1529267,
 1529268,
 1529269,
 1529270,
 1529271,
 1529272,
 1529273,
 1529274,
 1529275,
 1529276,
 1529277,
 1529278,
 1529279,
 1529280,
 1529281,
 1529282,
 1529283,
 1529284,
 1529285,
 1529286,
 1529287,
 1529288,
 1529289,
 1529290,
 1529291,
 1529292,
 1529293,
 1529294,
 1529295,
 1529296,
 1529297,
 1529298,
 1529299,
 1529300,
 1529301,
 1529302,
 1529303,
 1529304,
 1529305,
 1529306,
 1529307,
 1529308,
 1529309,
 1529310,
 1529311,
 1529312,
 1529313,
 1535462,
 1535463,
 1535464,
 1535465]

In [5]:
reg_season_ids

[1527674,
 1527675,
 1527676,
 1527677,
 1527678,
 1527679,
 1527680,
 1527681,
 1527682,
 1527683,
 1527684,
 1527685,
 1527686,
 1527687,
 1527688,
 1527689,
 1527690,
 1527691,
 1527692,
 1527693,
 1529264,
 1529265,
 1529266,
 1529267,
 1529268,
 1529269,
 1529270,
 1529271,
 1529272,
 1529273,
 1529274,
 1529275,
 1529276,
 1529277,
 1529278,
 1529279,
 1529280,
 1529281,
 1529282,
 1529283,
 1529284,
 1529285,
 1529286,
 1529287,
 1529288,
 1529289,
 1529290,
 1529291,
 1529292,
 1529293,
 1529294,
 1529295,
 1529296,
 1529297,
 1529298,
 1529299,
 1529300,
 1529301,
 1529302,
 1529303,
 1529304,
 1529305,
 1529306,
 1529307,
 1529308,
 1529309,
 1529310,
 1529311,
 1529312,
 1529313]

In [6]:
playoff_ids

[1535462, 1535463, 1535464, 1535465]

### Get 2026 Match Reg Season Info

In [7]:
# -------------------------------------------------------
# STEP 2: Get match and player info for reg season
# -------------------------------------------------------
def load_match_info(folder, match_ids):
    """
    Reads .json for each 2026 match and returns:
      - matches_df : one row per match
      - players_df : unique player registry (name + Cricsheet ID)
      - officials_df: unique official registry
    """
    all_matches = []
    all_registry = []
    all_players = []
    all_officials = []
 
    for match_id in match_ids:
        filename = f"{match_id}.json"
        filepath = os.path.join(folder, filename)
        if not os.path.exists(filepath):
            print(f"  Warning: {filename} not found, skipping")
            continue
 
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                d = json.load(f)
 
            # --- Get registry info ------
            registry_people = d.get("info", {}).get("registry", {}).get("people",{})
 
            # --- Retrieve data for names and Cricsheet ID ---
            for player_name, cricsheet_id in registry_people.items():
                if player_name and cricsheet_id:
                    all_registry.append({
                        "player_name": player_name,
                        "id": cricsheet_id
                    })


            # ---Get player info ----
            player_data = d.get("info", {}).get("players", {})
            for team, player_list in player_data.items():
                if team and player_list:
                    for player in player_list:
                        all_players.append({
                            "player_name": player,
                            "team": team
                        })

            # --- Get match info ----
            match_data = d.get("info", {})
            match_city = match_data.get("city", "")
            match_dates = match_data.get("dates", [])
            match_date = match_dates[0] if match_dates else ""
            match_number = match_data.get("event", {}).get("match_number", str(0))

            outcome_data = match_data.get("outcome", {})
            match_winner = ""
            match_outcome = ""
            
            if "winner" in outcome_data:
                match_winner = outcome_data.get("winner", "")
                match_outcome = outcome_data.get("by", {})
                if "wickets" in match_outcome:
                    match_outcome_w_r = "wickets"
                    match_outcome_amt = match_outcome.get("wickets", str(0))
                elif "runs" in match_outcome:
                    match_outcome_w_r = "runs"
                    match_outcome_amt = match_outcome.get("runs", str(0))
            elif "eliminator" in outcome_data:
                match_winner = outcome_data.get("eliminator", "")
                match_outcome = outcome_data.get("result", "")
                match_outcome_w_r = ""
                match_outcome_amt = ""
            elif "result" in outcome_data:
                match_winner = "NR"
                match_outcome = outcome_data.get("result", "")
                match_outcome_w_r = ""
                match_outcome_amt = ""


            # --- Teams Logic ---
            teams = match_data.get("teams", [])
            team_1 = teams[0] if len(teams) > 0 else ""
            team_2 = teams[1] if len(teams) > 1 else ""

            match_toss_winner = match_data.get("toss", {}).get("winner", "")
            match_toss_decision = match_data.get("toss", {}).get("decision", "")
            match_venue = match_data.get("venue", "")

            all_matches.append({
                "match_id": match_id,
                "city": match_city,
                "date": match_date,
                "number": match_number,
                "winner": match_winner,
                "outcome": match_outcome,
                "wickets_runs": match_outcome_w_r,
                "winning_amount": match_outcome_amt,
                "toss_winner": match_toss_winner,
                "toss_decision": match_toss_decision,
                "venue": match_venue,
                "team1": team_1,
                "team2": team_2

            })
            
            #Get official info
            official_data = match_data.get("officials", {})
            match_referees = official_data.get("match_referees", [])
            match_referee = match_referees[0] if match_referees else ""
            reserve_umpires = official_data.get("reserve_umpires", [])
            reserve_umpire = reserve_umpires[0] if reserve_umpires else ""
            tv_umpires = official_data.get("tv_umpires", [])
            tv_umpire = tv_umpires[0] if tv_umpires else ""
            umpires = official_data.get("umpires", [])
            if umpires:
                umpire1 = umpires[0] 
                umpire2 = umpires[1]
                
            else:
                umpire1 = ""
                umpire2 = ""

            all_officials.append({
                "match_id": match_id,
                "match_referee": match_referee,
                "reserve_umpire": reserve_umpire,
                "tv_umpire": tv_umpire,
                "first_umpire": umpire1,
                "second_umpire": umpire2
            })
        
        except Exception as e:
            print(f"  Warning: could not load {filename} — {e}")
 
    matches_df = (pd.DataFrame(all_matches))
    #print(matches_df)
    officials_df = (pd.DataFrame(all_officials))
    registry_df = (pd.DataFrame(all_registry).drop_duplicates(subset="player_name"))
    #print(registry_df)                
    players_df = (pd.DataFrame(all_players).drop_duplicates(subset="player_name"))
    #print(team_df)
    players_id_df = (pd.merge(registry_df, players_df, on="player_name", how='left').sort_values("team").reset_index(drop=True))
 
    print(f"✓ Matches loaded:  {len(matches_df)}")
    print(f"✓ Players found:   {len(players_df)}")
    return matches_df, players_id_df, registry_df

In [8]:
def load_delivery_files(folder, match_ids):
    """
    Reads the ball-by-ball delivery CSV for each 2026 match.
    
    """
    all_deliveries = []
 
    #print(f"\nLoading delivery files...")
 
    for match_id in match_ids:
        filename = f"{match_id}.json"
        filepath = os.path.join(folder, filename)
 
        if not os.path.exists(filepath):
            print(f"  Warning: {filename} not found, skipping")
            continue
 
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                d = json.load(f)

            innings_data = d.get("innings", [])
            for innings in innings_data[:2]:
                team = innings.get("team", "")

                powerplays_data = innings.get("powerplays", [])
                powerplays_from = 0.0
                powerplays_to = 0.0
                powerplays_type = ""
                if powerplays_data:
                    powerplays_from = powerplays_data[0].get("from", 0.0)
                    powerplays_to = powerplays_data[0].get("to", 0.0)
                    powerplays_type = powerplays_data[0].get("type", "")

                for overs_data in innings.get("overs", []):
                    over = overs_data.get("over", -1)

                    for delivery in overs_data.get("deliveries", []):
                        actual_delivery = delivery.get("actual_delivery", "")
                        batsman = delivery.get("batter", "")
                        bowler = delivery.get("bowler", "")
                        extra_data = delivery.get("extras", {})
                        extra_type = ""
                        extra_value = 0
                        if extra_data:
                            for extra, value in extra_data.items():
                                extra_type = extra
                                
                        non_striker = delivery.get("non_striker", "")
                        batter_runs = delivery.get("runs", {}).get("batter", "")
                        extra_runs = delivery.get("runs", {}).get("extras", "")
                        total_runs = delivery.get("runs", {}).get("total", "")
                        review_by = delivery.get("review", {}).get("by", "")
                        review_umpire = delivery.get("review", {}).get("umpire", "")
                        review_batter = delivery.get("review", {}).get("batter", "")
                        review_decision = delivery.get("review", {}).get("decision", "")
                        review_type = delivery.get("review", {}).get("type", "")
                        wickets_data_list = delivery.get("wickets", [])
                        player_out = ""
                        wicket_fielder = []
                        wicket_fielder_name = ""
                        wicket_kind = ""
                        if wickets_data_list:
                            player_out = wickets_data_list[0].get("player_out", "")
                            wicket_fielder = wickets_data_list[0].get("fielders", [])
                            if wicket_fielder:
                                wicket_fielder_name = wicket_fielder[0].get("name", "")
                            wicket_kind = wickets_data_list[0].get("kind", "")
                        if (float(actual_delivery) >= powerplays_from) and (float(actual_delivery) <= powerplays_to):                       
                            powerplay_flag = 1
                            current_powerplay_type = powerplays_type
                        else:
                            powerplay_flag = 0
                            current_powerplay_type = ""
                        


                        all_deliveries.append({
                            "match_id":match_id,
                            "team": team,
                            "over": over,
                            "actual_delivery": actual_delivery,
                            "batsman": batsman,
                            "bowler": bowler,
                            "non_striker": non_striker,
                            "batter_runs": batter_runs,
                            "extra_type": extra_type,
                            "extra_runs": extra_runs,
                            "total_runs": total_runs,
                            "review_by": review_by,
                            "review_umpire": review_umpire,
                            "review_batter": review_batter,
                            "review_decision": review_decision,
                            "review_type": review_type,
                            "player_out": player_out,
                            "wicket_fielder": wicket_fielder_name,
                            "wicket_kind": wicket_kind,
                            "powerplay_flag": powerplay_flag,
                            "powerplays_type": current_powerplay_type
                                })
 
        except Exception as e:
            print(f"  Warning: could not load {filename} — {e}")
 
    deliveries_df = pd.DataFrame(all_deliveries)
    print(f"✓ Total deliveries loaded: {len(deliveries_df):,}")
    return deliveries_df

In [9]:
display(len(match_ids))

74

In [10]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
matches_df, players_df, registry_df= load_match_info(IPL_DATA_FOLDER, reg_season_ids)
matches_df

✓ Matches loaded:  70
✓ Players found:   201


,match_id,city,date,number,winner,outcome,wickets_runs,winning_amount,toss_winner,toss_decision,venue,team1,team2
0,1527674,Bengaluru,2026-03-28,1,Royal Challengers Bengaluru,{'wickets': 6},wickets,6,Royal Challengers Bengaluru,field,"M Chinnaswamy Stadium, Bengaluru",Sunrisers Hyderabad,Royal Challengers Bengaluru
1,1527675,Mumbai,2026-03-29,2,Mumbai Indians,{'wickets': 6},wickets,6,Mumbai Indians,field,"Wankhede Stadium, Mumbai",Kolkata Knight Riders,Mumbai Indians
2,1527676,Guwahati,2026-03-30,3,Rajasthan Royals,{'wickets': 8},wickets,8,Rajasthan Royals,field,"Barsapara Cricket Stadium, Guwahati",Chennai Super Kings,Rajasthan Royals
3,1527677,New Chandigarh,2026-03-31,4,Punjab Kings,{'wickets': 3},wickets,3,Punjab Kings,field,Maharaja Yadavindra Singh International Cricke...,Gujarat Titans,Punjab Kings
4,1527678,Lucknow,2026-04-01,5,Delhi Capitals,{'wickets': 6},wickets,6,Delhi Capitals,field,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,Delhi Capitals
5,1527679,Kolkata,2026-04-02,6,Sunrisers Hyderabad,{'runs': 65},runs,65,Kolkata Knight Riders,field,"Eden Gardens, Kolkata",Sunrisers Hyderabad,Kolkata Knight Riders
6,1527680,Chennai,2026-04-03,7,Punjab Kings,{'wickets': 5},wickets,5,Punjab Kings,field,"MA Chidambaram Stadium, Chepauk, Chennai",Chennai Super Kings,Punjab Kings
7,1527681,Delhi,2026-04-04,8,Delhi Capitals,{'wickets': 6},wickets,6,Delhi Capitals,field,"Arun Jaitley Stadium, Delhi",Mumbai Indians,Delhi Capitals
8,1527682,Ahmedabad,2026-04-04,9,Rajasthan Royals,{'runs': 6},runs,6,Rajasthan Royals,bat,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans
9,1527683,Hyderabad,2026-04-05,10,Lucknow Super Giants,{'wickets': 5},wickets,5,Lucknow Super Giants,field,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Sunrisers Hyderabad,Lucknow Super Giants


In [11]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(players_df)

,player_name,id,team
0,A Kamboj,fcc21ace,Chennai Super Kings
1,A Mhatre,b2b4f545,Chennai Super Kings
2,J Overton,59559bc2,Chennai Super Kings
3,KK Ahmed,a2f46292,Chennai Super Kings
4,Kartik Sharma,7b679de5,Chennai Super Kings
5,MJ Henry,e84ac20c,Chennai Super Kings
6,MW Short,a90e53ec,Chennai Super Kings
7,Noor Ahmad,efc04be7,Chennai Super Kings
8,RD Gaikwad,45a43fe2,Chennai Super Kings
9,S Dube,a4e37e47,Chennai Super Kings


In [12]:
officials_df = players_df[players_df['team'].isna()]
officials_df

,player_name,id,team
201,A Bengeri,c109497d,NaN
202,J Madanagopal,8275b04a,NaN
203,J Srinath,bad31fac,NaN
204,R Pandit,8a604384,NaN
205,UV Gandhe,43dd4011,NaN
206,Anish Sahasrabudhe,2892df67,NaN
207,MV Saidharshan Kumar,54b6007b,NaN
208,Nitin Menon,e1d41d9e,NaN
209,Shakti Singh,eb90d319,NaN
210,Vinod Seshan,a3357a27,NaN


In [13]:
players_df = players_df[(players_df['team']).notna()]
players_df

,player_name,id,team
0,A Kamboj,fcc21ace,Chennai Super Kings
1,A Mhatre,b2b4f545,Chennai Super Kings
2,J Overton,59559bc2,Chennai Super Kings
3,KK Ahmed,a2f46292,Chennai Super Kings
4,Kartik Sharma,7b679de5,Chennai Super Kings
5,MJ Henry,e84ac20c,Chennai Super Kings
6,MW Short,a90e53ec,Chennai Super Kings
7,Noor Ahmad,efc04be7,Chennai Super Kings
8,RD Gaikwad,45a43fe2,Chennai Super Kings
9,S Dube,a4e37e47,Chennai Super Kings


In [14]:
deliveries_df = load_delivery_files(IPL_DATA_FOLDER, reg_season_ids)
deliveries_df.info()


✓ Total deliveries loaded: 16,548
<class 'pandas.DataFrame'>
RangeIndex: 16548 entries, 0 to 16547
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   match_id         16548 non-null  int64
 1   team             16548 non-null  str  
 2   over             16548 non-null  int64
 3   actual_delivery  16548 non-null  str  
 4   batsman          16548 non-null  str  
 5   bowler           16548 non-null  str  
 6   non_striker      16548 non-null  str  
 7   batter_runs      16548 non-null  int64
 8   extra_type       16548 non-null  str  
 9   extra_runs       16548 non-null  int64
 10  total_runs       16548 non-null  int64
 11  review_by        16548 non-null  str  
 12  review_umpire    16548 non-null  str  
 13  review_batter    16548 non-null  str  
 14  review_decision  16548 non-null  str  
 15  review_type      16548 non-null  str  
 16  player_out       16548 non-null  str  
 17  wicket_fielder   16548 non-

In [46]:
def calculate_batting_stats(deliveries_df, players_df):
     """
    Calculates overall season batting stats for every player from the deliveries DataFrame.
    Returns a single player_stats_df with one row per player containing:
      - Batting stats
    All joined to the players table on player_name.

    Corrections applied:
      - Wides excluded from balls faced (no-balls count as faced)
      - Dismissals counted via player_dismissed (catches non-striker run outs too)
    """
     
      # -------------------------------------------------------
    # BATTING STATS
    # grouped by striker (the batter facing the ball)
    # -------------------------------------------------------

    # A ball counts as faced only if it's not a wide
    # No-balls count as faced — batter can play the ball
     deliveries_df["ball_faced"] = (deliveries_df["extra_type"] != "wides").astype(int)

    # Boundary fours and sixes
     deliveries_df["is_four"] = (deliveries_df["batter_runs"] == 4).astype(int)
     deliveries_df["is_six"]  = (deliveries_df["batter_runs"] == 6).astype(int)

     #Dot balls
     deliveries_df["is_dot_ball"] = (deliveries_df["total_runs"] == 0).astype(int)

     #Separate deliveries of innings into Powerplay, Middle Period and Death Overs
     

    # Assign clean flags that completely ignore individual wide/no-ball row counts
     deliveries_df["is_death"] = (
    (deliveries_df["over"] >= 15) & 
    (deliveries_df["powerplay_flag"] == 0)
     ).astype(int)

     deliveries_df["is_middle"] = (
    (deliveries_df["powerplay_flag"] == 0) & 
    (deliveries_df["is_death"] == 0)
     ).astype(int)

         

     ## ----------------------------------------------------
     ## POWERPLAY BATTING STATS
     ## ----------------------------------------------------
     pp_batting_stats = deliveries_df[deliveries_df["powerplay_flag"] == 1].groupby("batsman").agg(
          pp_runs      = ("batter_runs",     "sum"),
          pp_balls_faced = ("ball_faced",    "sum"),
          pp_dot_balls   = ("is_dot_ball",   "sum"),
          pp_fours       = ("is_four",       "sum"),
          pp_sixes       = ("is_six",        "sum"),
          pp_innings_batted = ("match_id",    "nunique")
     ).reset_index().rename(columns= {"batsman": "player_name"})

     ## ----------------------------------------------------
     ## MIDDLE OVERS BATTING STATS
     ## ----------------------------------------------------
     middle_batting_stats = deliveries_df[deliveries_df["is_middle"] == 1].groupby("batsman").agg(
          middle_runs          = ("batter_runs",     "sum"),
          middle_balls_faced   = ("ball_faced",      "sum"),
          middle_dot_balls     = ("is_dot_ball",     "sum"),
          middle_fours         = ("is_four",         "sum"),
          middle_sixes         = ("is_six",          "sum"),
          middle_innings_batted = ("match_id",       "nunique")
     ).reset_index().rename(columns={"batsman": "player_name"})
     
     ## ----------------------------------------------------
     ## DEATH OVERS BATTING STATS
     ## ----------------------------------------------------
     death_batting_stats = deliveries_df[deliveries_df["is_death"] == 1].groupby("batsman").agg(
          death_runs          = ("batter_runs",     "sum"),
          death_balls_faced   = ("ball_faced",      "sum"),
          death_dot_balls     = ("is_dot_ball",     "sum"),
          death_fours         = ("is_four",         "sum"),
          death_sixes         = ("is_six",          "sum"),
          death_innings_batted = ("match_id",       "nunique")
     ).reset_index().rename(columns={"batsman": "player_name"})

     ## -----------------------------------------------------
     ## INNINGS PERIOD BATTING STATS
     ## -----------------------------------------------------
     master_batting_df = pp_batting_stats.merge(middle_batting_stats, on="player_name", how="outer")
     master_batting_df = master_batting_df.merge(death_batting_stats, on="player_name", how="outer")
     master_batting_df.fillna(0, inplace=True)

     ## STRIKE RATE FOR EACH BATSMAN FOR EACH PHASE
     master_batting_df["pp_strike_rate"] = (master_batting_df["pp_runs"] / master_batting_df["pp_balls_faced"] * 100).fillna(0).round(2)
     master_batting_df["middle_strike_rate"] = (master_batting_df["middle_runs"] / master_batting_df["middle_balls_faced"] * 100).fillna(0).round(2)
     master_batting_df["death_strike_rate"] = (master_batting_df["death_runs"] / master_batting_df["death_balls_faced"] * 100).fillna(0).round(2)

     ## BATTING BOUNDARY PERCENTAGE FOR EACH BATSMAN EACH PHASE
     master_batting_df["pp_boundary_pct"] = ((master_batting_df["pp_fours"] + master_batting_df["pp_sixes"]) / master_batting_df["pp_balls_faced"] * 100).fillna(0).round(2)
     master_batting_df["middle_boundary_pct"] = ((master_batting_df["middle_fours"] + master_batting_df["middle_sixes"]) / master_batting_df["middle_balls_faced"] * 100).fillna(0).round(2)
     master_batting_df["death_boundary_pct"] = ((master_batting_df["death_fours"] + master_batting_df["death_sixes"]) / master_batting_df["death_balls_faced"] * 100).fillna(0).round(2)

     ## DOT BALL PERCENTAGE FOR EACH BATSMAN FOR EACH PHASE
     master_batting_df["pp_dot_ball_pct"] = (master_batting_df["pp_dot_balls"] / master_batting_df['pp_balls_faced'] *100).fillna(0).round(2)
     master_batting_df["middle_dot_ball_pct"] = (master_batting_df["middle_dot_balls"] / master_batting_df['middle_balls_faced'] *100).fillna(0).round(2)
     master_batting_df["death_dot_ball_pct"] = (master_batting_df["death_dot_balls"] / master_batting_df['death_balls_faced'] *100).fillna(0).round(2)

     
     #Non boundary Strike Rate - PowerPlay
     master_batting_df["pp_non_boundary_runs"] = (master_batting_df["pp_runs"] - ((4 * master_batting_df["pp_fours"]) + (6 * master_batting_df["pp_sixes"]))).fillna(0)
     master_batting_df["pp_non_boundary_balls"] =(master_batting_df["pp_balls_faced"] -(master_batting_df["pp_fours"] + master_batting_df["pp_sixes"])).fillna(0)
     master_batting_df["pp_non_boundary_strike_rate"] = (master_batting_df["pp_non_boundary_runs"] / master_batting_df["pp_non_boundary_balls"] * 100).fillna(0).round(2)

     #Non boundary Strike Rate - Middle Overs
     master_batting_df["middle_non_boundary_runs"] = (master_batting_df["middle_runs"] - ((4 * master_batting_df["middle_fours"]) + (6 * master_batting_df["middle_sixes"]))).fillna(0)
     master_batting_df["middle_non_boundary_balls"] =(master_batting_df["middle_balls_faced"] -(master_batting_df["middle_fours"] + master_batting_df["middle_sixes"])).fillna(0)
     master_batting_df["middle_non_boundary_strike_rate"] = (master_batting_df["middle_non_boundary_runs"] / master_batting_df["middle_non_boundary_balls"] *100).fillna(0).round(2)

     #Non boundary Strike Rate - Death Overs
     master_batting_df["death_non_boundary_runs"] = (master_batting_df["death_runs"] - ((4 * master_batting_df["death_fours"]) + (6 * master_batting_df["death_sixes"]))).fillna(0)
     master_batting_df["death_non_boundary_balls"] =(master_batting_df["death_balls_faced"] -(master_batting_df["death_fours"] + master_batting_df["death_sixes"])).fillna(0)
     master_batting_df["death_non_boundary_strike_rate"] = (master_batting_df["death_non_boundary_runs"] / master_batting_df["death_non_boundary_balls"] * 100).fillna(0).round(2)

     ## --------------------------------------------------
     ## OVERALL BATTING STATS
     ## ----------------------------------------------------
     batting_stats = deliveries_df.groupby("batsman").agg(
        runs           = ("batter_runs",  "sum"),
        balls_faced    = ("ball_faced",    "sum"),
        dot_balls      = ("is_dot_ball",   "sum"),
        fours          = ("is_four",       "sum"),
        sixes          = ("is_six",        "sum"),
        innings_batted = ("match_id",      "nunique")
    ).reset_index().rename(columns={"batsman": "player_name"})
     
     # Dismissals — grouped by player_dismissed so non-striker run outs are included
     dismissals = (deliveries_df[(deliveries_df["player_out"].notna()) & (deliveries_df["player_out"] != "")]
                    .groupby("player_out")
                    .size()
                    .reset_index())
     dismissals.columns = ["player_name", "dismissals"]

     batting_stats = batting_stats.merge(dismissals, on="player_name", how="left")
     batting_stats["dismissals"] = batting_stats["dismissals"].fillna(0).astype(int)

    # Strike rate = (runs / balls faced) * 100
     batting_stats["strike_rate"] = (
        batting_stats["runs"] / batting_stats["balls_faced"] * 100
    ).round(2)

    # Batting average = runs / dismissals (NaN if never dismissed)
     batting_stats["batting_average"] = (
        batting_stats["runs"] / batting_stats["dismissals"].replace(0, float("nan"))
    ).round(2)
     
    # Batting boundary percentage = (fours + sixes) / balls_faced * 100
     batting_stats["boundary_pct"] = ((batting_stats["fours"] + batting_stats["sixes"]) / batting_stats["balls_faced"] * 100).round(2)
     
     # Dot ball percentage
     batting_stats["dot_ball_pct"] = (batting_stats["dot_balls"] / batting_stats['balls_faced'] * 100).round(2)

     #Non boundary Strike Rate
     batting_stats["non_boundary_runs"] = (batting_stats["runs"] - ((4 * batting_stats["fours"]) + (6 * batting_stats["sixes"])))
     batting_stats["non_boundary_balls"] =(batting_stats["balls_faced"] -(batting_stats["fours"] + batting_stats["sixes"]))
     batting_stats["non_boundary_strike_rate"] = (batting_stats["non_boundary_runs"] / batting_stats["non_boundary_balls"] * 100).round(2)

     
     player_batting_stats_df = (players_df
        .merge(batting_stats,  on="player_name", how="left"))
     player_batting_stats_df = (player_batting_stats_df.merge(master_batting_df, on="player_name", how="outer"))
     
     print(f"✓ Player stats calculated: {len(player_batting_stats_df)} players")
     print(f"  Columns: {list(player_batting_stats_df.columns)}")

     return player_batting_stats_df

In [47]:
player_batting_stats_df = calculate_batting_stats(deliveries_df=deliveries_df, players_df=players_df)
player_batting_stats_df= player_batting_stats_df[(player_batting_stats_df["team"].notna())]
player_batting_stats_df


✓ Player stats calculated: 201 players
  Columns: ['player_name', 'id', 'team', 'runs', 'balls_faced', 'dot_balls', 'fours', 'sixes', 'innings_batted', 'dismissals', 'strike_rate', 'batting_average', 'boundary_pct', 'dot_ball_pct', 'non_boundary_runs', 'non_boundary_balls', 'non_boundary_strike_rate', 'pp_runs', 'pp_balls_faced', 'pp_dot_balls', 'pp_fours', 'pp_sixes', 'pp_innings_batted', 'middle_runs', 'middle_balls_faced', 'middle_dot_balls', 'middle_fours', 'middle_sixes', 'middle_innings_batted', 'death_runs', 'death_balls_faced', 'death_dot_balls', 'death_fours', 'death_sixes', 'death_innings_batted', 'pp_strike_rate', 'middle_strike_rate', 'death_strike_rate', 'pp_boundary_pct', 'middle_boundary_pct', 'death_boundary_pct', 'pp_dot_ball_pct', 'middle_dot_ball_pct', 'death_dot_ball_pct', 'pp_non_boundary_runs', 'pp_non_boundary_balls', 'pp_non_boundary_strike_rate', 'middle_non_boundary_runs', 'middle_non_boundary_balls', 'middle_non_boundary_strike_rate', 'death_non_boundary_runs

,player_name,id,team,runs,balls_faced,dot_balls,fours,sixes,innings_batted,dismissals,strike_rate,batting_average,boundary_pct,dot_ball_pct,non_boundary_runs,non_boundary_balls,non_boundary_strike_rate,pp_runs,pp_balls_faced,pp_dot_balls,pp_fours,pp_sixes,pp_innings_batted,middle_runs,middle_balls_faced,middle_dot_balls,middle_fours,middle_sixes,middle_innings_batted,death_runs,death_balls_faced,death_dot_balls,death_fours,death_sixes,death_innings_batted,pp_strike_rate,middle_strike_rate,death_strike_rate,pp_boundary_pct,middle_boundary_pct,death_boundary_pct,pp_dot_ball_pct,middle_dot_ball_pct,death_dot_ball_pct,pp_non_boundary_runs,pp_non_boundary_balls,pp_non_boundary_strike_rate,middle_non_boundary_runs,middle_non_boundary_balls,middle_non_boundary_strike_rate,death_non_boundary_runs,death_non_boundary_balls,death_non_boundary_strike_rate
0,A Badoni,872b03f7,Lucknow Super Giants,215.0,141.0,48.0,23.0,9.0,10.0,10.0,152.48,21.50,22.70,34.04,69.0,109.0,63.30,95.0,52.0,20.0,12.0,5.0,6.0,96.0,75.0,23.0,10.0,2.0,7.0,24.0,14.0,5.0,1.0,2.0,3.0,182.69,128.00,171.43,32.69,16.00,21.43,38.46,30.67,35.71,17.0,35.0,48.57,44.0,63.0,69.84,8.0,11.0,72.73
1,A Kamboj,fcc21ace,Chennai Super Kings,58.0,38.0,6.0,3.0,4.0,4.0,1.0,152.63,58.00,18.42,15.79,22.0,31.0,70.97,0.0,0.0,0.0,0.0,0.0,0.0,19.0,8.0,2.0,1.0,2.0,1.0,39.0,30.0,4.0,2.0,2.0,3.0,0.00,237.50,130.00,0.00,37.50,13.33,0.00,25.00,13.33,0.0,0.0,0.00,3.0,5.0,60.00,19.0,26.0,73.08
2,A Mhatre,b2b4f545,Chennai Super Kings,201.0,113.0,38.0,20.0,12.0,6.0,6.0,177.88,33.50,28.32,33.63,49.0,81.0,60.49,98.0,53.0,24.0,15.0,4.0,5.0,97.0,53.0,13.0,5.0,8.0,2.0,6.0,7.0,1.0,0.0,0.0,1.0,184.91,183.02,85.71,35.85,24.53,0.00,45.28,24.53,14.29,14.0,34.0,41.18,29.0,40.0,72.50,6.0,7.0,85.71
3,A Nortje,acdc62f5,Lucknow Super Giants,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.00,0.00,0.00,100.00,0.0,1.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,100.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,1.0,0.00
4,A Raghuvanshi,d7017798,Kolkata Knight Riders,422.0,288.0,93.0,40.0,19.0,12.0,10.0,146.53,42.20,20.49,32.29,148.0,229.0,64.63,129.0,102.0,48.0,12.0,7.0,11.0,214.0,146.0,39.0,20.0,8.0,8.0,79.0,40.0,6.0,8.0,4.0,4.0,126.47,146.58,197.50,18.63,19.18,30.00,47.06,26.71,15.00,39.0,83.0,46.99,86.0,118.0,72.88,23.0,28.0,82.14
5,AA Kulkarni,2d140b79,Lucknow Super Giants,17.0,25.0,14.0,1.0,0.0,2.0,2.0,68.00,8.50,4.00,56.00,13.0,24.0,54.17,8.0,13.0,8.0,0.0,0.0,2.0,9.0,12.0,6.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,61.54,75.00,0.00,0.00,8.33,0.00,61.54,50.00,0.00,8.0,13.0,61.54,5.0,11.0,45.45,0.0,0.0,0.00
6,AF Milne,350bb1b1,Rajasthan Royals,2.0,2.0,1.0,0.0,0.0,1.0,0.0,100.00,NaN,0.00,50.00,2.0,2.0,100.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,1.0,0.0,0.0,1.0,0.00,0.00,100.00,0.00,0.00,0.00,0.00,0.00,50.00,0.0,0.0,0.00,0.0,0.0,0.00,2.0,2.0,100.00
7,AJ Hosein,4d7f517e,Chennai Super Kings,5.0,3.0,0.0,0.0,0.0,2.0,0.0,166.67,NaN,0.00,0.00,5.0,3.0,166.67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,3.0,0.0,0.0,0.0,2.0,0.00,0.00,166.67,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.00,5.0,3.0,166.67
8,AK Markram,6a26221c,Lucknow Super Giants,231.0,167.0,52.0,20.0,11.0,11.0,9.0,138.32,25.67,18.56,31.14,85.0,136.0,62.50,120.0,87.0,35.0,13.0,6.0,7.0,57.0,52.0,12.0,3.0,1.0,6.0,54.0,28.0,5.0,4.0,4.0,3.0,137.93,109.62,192.86,21.84,7.69,28.57,40.23,23.08,17.86,32.0,68.0,47.06,39.0,48.0,81.25,14.0,20.0,70.00
9,AM Ghazanfar,9ca68676,Mumbai Indians,2.0,3.0,1.0,0.0,0.0,1.0,0.0,66.67,NaN,0.00,33.33,2.0,3.0,66.67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0,1.0,0.0,0.0,1.0,0.00,0.00,66.67,0.00,0.00,0.00,0.00,0.00,33.33,0.0,0.0,0.00,0.0,0.0,0.00,2.0,3.0,66.67


In [48]:
player_batting_stats_df[player_batting_stats_df["player_name"].str.contains("Suryavanshi")]

,player_name,id,team,runs,balls_faced,dot_balls,fours,sixes,innings_batted,dismissals,strike_rate,batting_average,boundary_pct,dot_ball_pct,non_boundary_runs,non_boundary_balls,non_boundary_strike_rate,pp_runs,pp_balls_faced,pp_dot_balls,pp_fours,pp_sixes,pp_innings_batted,middle_runs,middle_balls_faced,middle_dot_balls,middle_fours,middle_sixes,middle_innings_batted,death_runs,death_balls_faced,death_dot_balls,death_fours,death_sixes,death_innings_batted,pp_strike_rate,middle_strike_rate,death_strike_rate,pp_boundary_pct,middle_boundary_pct,death_boundary_pct,pp_dot_ball_pct,middle_dot_ball_pct,death_dot_ball_pct,pp_non_boundary_runs,pp_non_boundary_balls,pp_non_boundary_strike_rate,middle_non_boundary_runs,middle_non_boundary_balls,middle_non_boundary_strike_rate,death_non_boundary_runs,death_non_boundary_balls,death_non_boundary_strike_rate
189,V Suryavanshi,470f446b,Rajasthan Royals,583.0,251.0,86.0,50.0,53.0,14.0,14.0,232.27,41.64,41.04,34.26,65.0,148.0,43.92,404.0,177.0,66.0,40.0,34.0,14.0,179.0,74.0,20.0,10.0,19.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,228.25,241.89,0.0,41.81,39.19,0.0,37.29,27.03,0.0,40.0,103.0,38.83,25.0,45.0,55.56,0.0,0.0,0.0


In [49]:
player_batting_stats_df[player_batting_stats_df["player_name"].str.contains("Pooran")]

,player_name,id,team,runs,balls_faced,dot_balls,fours,sixes,innings_batted,dismissals,strike_rate,batting_average,boundary_pct,dot_ball_pct,non_boundary_runs,non_boundary_balls,non_boundary_strike_rate,pp_runs,pp_balls_faced,pp_dot_balls,pp_fours,pp_sixes,pp_innings_batted,middle_runs,middle_balls_faced,middle_dot_balls,middle_fours,middle_sixes,middle_innings_batted,death_runs,death_balls_faced,death_dot_balls,death_fours,death_sixes,death_innings_batted,pp_strike_rate,middle_strike_rate,death_strike_rate,pp_boundary_pct,middle_boundary_pct,death_boundary_pct,pp_dot_ball_pct,middle_dot_ball_pct,death_dot_ball_pct,pp_non_boundary_runs,pp_non_boundary_balls,pp_non_boundary_strike_rate,middle_non_boundary_runs,middle_non_boundary_balls,middle_non_boundary_strike_rate,death_non_boundary_runs,death_non_boundary_balls,death_non_boundary_strike_rate
111,N Pooran,3241e3fd,Lucknow Super Giants,234.0,183.0,81.0,11.0,19.0,14.0,13.0,127.87,18.0,16.39,44.26,76.0,153.0,49.67,49.0,37.0,19.0,2.0,5.0,5.0,144.0,131.0,59.0,8.0,9.0,13.0,41.0,15.0,3.0,1.0,5.0,2.0,132.43,109.92,273.33,18.92,12.98,40.0,51.35,45.04,20.0,11.0,30.0,36.67,58.0,114.0,50.88,7.0,9.0,77.78


In [50]:
deliveries_df["wicket_kind"].unique()

<ArrowStringArray>
[                     '',                'caught',               'run out',
                'bowled',                   'lbw',     'caught and bowled',
               'stumped',           'retired out',          'retired hurt',
 'obstructing the field']
Length: 10, dtype: str

In [51]:
def calculate_bowling_stats(deliveries_df, players_df):
    """
    Calculates overall season bowling stats for every player from the deliveries DataFrame.
    Returns a single player_stats_df with one row per player containing:
      - Bowling stats
    All joined to the players table on player_name.

    Corrections applied:
      - Runs conceded includes wides and no-balls (but not byes or leg byes)
    """

    # -------------------------------------------------------
    # BOWLING STATS
    # grouped by bowler
    # -------------------------------------------------------

    # A legal delivery excludes wides and no-balls
    deliveries_df["is_legal"] = (
        (deliveries_df["extra_type"] != "wides") & (deliveries_df["extra_type"] != "noballs")
    ).astype(int)

    deliveries_df["wides"] = np.where(deliveries_df["extra_type"] == "wides", deliveries_df["extra_runs"], 0)
    deliveries_df["noballs"] = np.where(deliveries_df["extra_type"] == "noballs", deliveries_df["extra_runs"], 0)

    # Runs conceded by bowler = runs off bat + wides + no-balls
    # Byes and leg byes are NOT the bowler's fault
    deliveries_df["runs_conceded"] = (
        deliveries_df["batter_runs"].fillna(0) +
        deliveries_df["wides"].fillna(0) +
        deliveries_df["noballs"].fillna(0)
    )

     # Wickets credited to bowler — excludes run outs, retired hurt, obstructions
    deliveries_df["bowler_wicket"] = (
        deliveries_df["wicket_kind"].notna() &
        ~deliveries_df["wicket_kind"].isin([
            "run out", "retired hurt", "retired out", "obstructing the field", ""
        ])
    ).astype(int)

    # Dot balls = legal delivery where total runs == 0
    deliveries_df["is_dot"] = (
        (deliveries_df["is_legal"] == 1) & (deliveries_df["total_runs"] == 0)
    ).astype(int)

    # Boundaries balls = delivery where runs off bat == 4 OR 6
    deliveries_df["is_boundary"] = (
        (deliveries_df["batter_runs"].isin([4, 6]))
    ).astype(int)

    bowling_stats = deliveries_df.groupby("bowler").agg(
        balls_bowled   = ("is_legal",        "sum"),
        runs_conceded  = ("runs_conceded",    "sum"),
        wickets        = ("bowler_wicket",    "sum"),
        dot_balls      = ("is_dot",           "sum"),
        boundaries     = ("is_boundary",      "sum"),
        innings_bowled = ("match_id",         "nunique")
    ).reset_index().rename(columns={"bowler": "player_name"})

    # Overs bowled (balls / 6, rounded to 1 decimal)
    bowling_stats["over"], bowling_stats["ball"] = bowling_stats["balls_bowled"].divmod(6)
    bowling_stats["overs_bowled"] = (bowling_stats["over"].astype(int).astype(str) + '.' + bowling_stats["ball"].astype(int).astype(str)).astype(float)

    # Economy rate = runs conceded per over
    bowling_stats["economy"] = (
        bowling_stats["runs_conceded"] / bowling_stats["balls_bowled"] * 6
    ).round(2)

    # Bowling average = runs conceded per wicket
    bowling_stats["bowling_average"] = (
        bowling_stats["runs_conceded"] / bowling_stats["wickets"].replace(0, float("nan"))
    ).round(2)

    # Bowling strike rate = balls bowled per wicket
    bowling_stats["bowling_strike_rate"] = (
    bowling_stats["balls_bowled"] / bowling_stats["wickets"].replace(0, float("nan"))
    ).round(2)

    # Boundary percentage = legal deliveries hit for 4 or 6
    bowling_stats["boundary_pct"] = (bowling_stats["boundaries"] / bowling_stats["balls_bowled"] * 100).round(2)

    # Balls per boundary  = (Boundary percentage/100)^(-1)
    bowling_stats["balls_per_boundary"] = (100 / bowling_stats["boundary_pct"]).astype(int)    

    #Separate deliveries of innings into Powerplay, Middle Period and Death Overs
    # Assign clean flags that completely ignore individual wide/no-ball row counts
    deliveries_df["is_death"] = (
    (deliveries_df["over"] >= 15) & 
    (deliveries_df["powerplay_flag"] == 0)
     ).astype(int)

    deliveries_df["is_middle"] = (
    (deliveries_df["powerplay_flag"] == 0) & 
    (deliveries_df["is_death"] == 0)
     ).astype(int)


    ## ----------------------------------------------------
    ## POWERPLAY BOWLING STATS
    ## ----------------------------------------------------
    pp_bowling_stats = deliveries_df[deliveries_df["powerplay_flag"] == 1].groupby("bowler").agg(
        pp_balls_bowled   = ("is_legal",        "sum"),
        pp_runs_conceded  = ("runs_conceded",    "sum"),
        pp_wickets        = ("bowler_wicket",    "sum"),
        pp_dot_balls      = ("is_dot",           "sum"),
        pp_boundaries     = ("is_boundary",      "sum"),
        pp_innings_bowled = ("match_id",         "nunique")
    ).reset_index().rename(columns={"bowler": "player_name"})

    # Overs bowled (balls / 6, rounded to 1 decimal) - Powerplay
    pp_bowling_stats["pp_over"], pp_bowling_stats["pp_ball"] = pp_bowling_stats["pp_balls_bowled"].divmod(6)
    pp_bowling_stats["pp_overs_bowled"] = (pp_bowling_stats["pp_over"].astype(int).astype(str) + '.' + pp_bowling_stats["pp_ball"].astype(int).astype(str)).astype(float)

    # Economy rate = runs conceded per over - Powerplay
    pp_bowling_stats["pp_economy"] = (
        pp_bowling_stats["pp_runs_conceded"] / pp_bowling_stats["pp_balls_bowled"] * 6
    ).round(2)

    # Bowling average = runs conceded per wicket - Powerplay
    pp_bowling_stats["pp_bowling_average"] = (
        pp_bowling_stats["pp_runs_conceded"] / pp_bowling_stats["pp_wickets"].replace(0, float("nan"))
    ).round(2)

    # Bowling strike rate = balls bowled per wicket - Powerplay
    pp_bowling_stats["pp_bowling_strike_rate"] = (
    pp_bowling_stats["pp_balls_bowled"] / pp_bowling_stats["pp_wickets"].replace(0, float("nan"))
    ).round(2)

     # Boundary percentage = legal deliveries hit for 4 or 6
    pp_bowling_stats["pp_boundary_pct"] = (pp_bowling_stats["pp_boundaries"] / pp_bowling_stats["pp_balls_bowled"] * 100).fillna(0).round(2)

    # Balls per boundary  = (Boundary percentage/100)^(-1)
    pp_bowling_stats["pp_balls_per_boundary"] = (
    (pp_bowling_stats["pp_balls_bowled"] / pp_bowling_stats["pp_boundaries"])
    .replace([float('inf'), float('-inf')], 0)
    .astype(int)
)

    ## ----------------------------------------------------
    ## MIDDLE PERIOD BOWLING STATS
    ## ----------------------------------------------------
    middle_bowling_stats = deliveries_df[deliveries_df["is_middle"] == 1].groupby("bowler").agg(
        middle_balls_bowled   = ("is_legal",        "sum"),
        middle_runs_conceded  = ("runs_conceded",    "sum"),
        middle_wickets        = ("bowler_wicket",    "sum"),
        middle_dot_balls      = ("is_dot",           "sum"),
        middle_boundaries     = ("is_boundary",      "sum"),
        middle_innings_bowled = ("match_id",         "nunique")
    ).reset_index().rename(columns={"bowler": "player_name"})

    # Overs bowled (balls / 6, rounded to 1 decimal) - Middle Overs
    middle_bowling_stats["middle_over"], middle_bowling_stats["middle_ball"] = middle_bowling_stats["middle_balls_bowled"].divmod(6)
    middle_bowling_stats["middle_overs_bowled"] = (middle_bowling_stats["middle_over"].astype(int).astype(str) + '.' + middle_bowling_stats["middle_ball"].astype(int).astype(str)).astype(float)

    # Economy rate = runs conceded per over - Middle Overs
    middle_bowling_stats["middle_economy"] = (
        middle_bowling_stats["middle_runs_conceded"] / middle_bowling_stats["middle_balls_bowled"] * 6
    ).round(2)

    # Bowling average = runs conceded per wicket - Middle Overs
    middle_bowling_stats["middle_bowling_average"] = (
        middle_bowling_stats["middle_runs_conceded"] / middle_bowling_stats["middle_wickets"].replace(0, float("nan"))
    ).round(2)

    # Bowling strike rate = balls bowled per wicket - Middle Overs
    middle_bowling_stats["middle_bowling_strike_rate"] = (
    middle_bowling_stats["middle_balls_bowled"] / middle_bowling_stats["middle_wickets"].replace(0, float("nan"))
    ).round(2)

     # Boundary percentage = legal deliveries hit for 4 or 6
    middle_bowling_stats["middle_boundary_pct"] = (middle_bowling_stats["middle_boundaries"] / middle_bowling_stats["middle_balls_bowled"] * 100).fillna(0).round(2)

    # Balls per boundary  = (Boundary percentage/100)^(-1)
    middle_bowling_stats["middle_balls_per_boundary"] = (
    (middle_bowling_stats["middle_balls_bowled"] / middle_bowling_stats["middle_boundaries"])
    .replace([float('inf'), float('-inf')], 0)
    .astype(int)
)

    ## ----------------------------------------------------
    ## DEATH PERIOD BOWLING STATS
    ## ----------------------------------------------------
    death_bowling_stats = deliveries_df[deliveries_df["is_death"] == 1].groupby("bowler").agg(
        death_balls_bowled   = ("is_legal",        "sum"),
        death_runs_conceded  = ("runs_conceded",    "sum"),
        death_wickets        = ("bowler_wicket",    "sum"),
        death_dot_balls      = ("is_dot",           "sum"),
        death_boundaries     = ("is_boundary",      "sum"),
        death_innings_bowled = ("match_id",         "nunique")
    ).reset_index().rename(columns={"bowler": "player_name"})

    # Overs bowled (balls / 6, rounded to 1 decimal) - Death Overs
    death_bowling_stats["death_over"], death_bowling_stats["death_ball"] = death_bowling_stats["death_balls_bowled"].divmod(6)
    death_bowling_stats["death_overs_bowled"] = (death_bowling_stats["death_over"].astype(int).astype(str) + '.' + death_bowling_stats["death_ball"].astype(int).astype(str)).astype(float)

    # Economy rate = runs conceded per over -  Death Overs
    death_bowling_stats["death_economy"] = (
        death_bowling_stats["death_runs_conceded"] / death_bowling_stats["death_balls_bowled"] * 6
    ).round(2)

    # Bowling average = runs conceded per wicket - Death Overs
    death_bowling_stats["death_bowling_average"] = (
        death_bowling_stats["death_runs_conceded"] / death_bowling_stats["death_wickets"].replace(0, float("nan"))
    ).round(2)

    # Bowling strike rate = balls bowled per wicket - Death Overs
    death_bowling_stats["death_bowling_strike_rate"] = (
    death_bowling_stats["death_balls_bowled"] / death_bowling_stats["death_wickets"].replace(0, float("nan"))
    ).round(2)

     # Boundary percentage = legal deliveries hit for 4 or 6
    death_bowling_stats["death_boundary_pct"] = (death_bowling_stats["death_boundaries"] / death_bowling_stats["death_balls_bowled"] * 100).fillna(0).round(2)

    # Balls per boundary  = (Boundary percentage/100)^(-1)
    death_bowling_stats["death_balls_per_boundary"] = (
    (death_bowling_stats["death_balls_bowled"] / death_bowling_stats["death_boundaries"])
    .replace([float('inf'), float('-inf')], 0)
    .astype(int)
)
    
    player_bowling_stats_df = (players_df
        .merge(bowling_stats,  on="player_name", how="left"))

    master_bowling_df = pp_bowling_stats.merge(middle_bowling_stats, on="player_name", how="outer")
    master_bowling_df = master_bowling_df.merge(death_bowling_stats, on="player_name", how="outer")
    master_bowling_df.fillna(0, inplace=True)

    player_bowling_stats_df = (player_bowling_stats_df.merge(master_bowling_df, on="player_name", how="outer"))
     
    print(f"✓ Player stats calculated: {len(player_bowling_stats_df)} players")
    print(f"  Columns: {list(player_bowling_stats_df.columns)}")

    return player_bowling_stats_df
    

In [52]:
player_bowling_stats_df = calculate_bowling_stats(deliveries_df=deliveries_df, players_df=players_df)
player_bowling_stats_df= player_bowling_stats_df[(player_bowling_stats_df["team"].notna())]
player_bowling_stats_df


✓ Player stats calculated: 201 players
  Columns: ['player_name', 'id', 'team', 'balls_bowled', 'runs_conceded', 'wickets', 'dot_balls', 'boundaries', 'innings_bowled', 'over', 'ball', 'overs_bowled', 'economy', 'bowling_average', 'bowling_strike_rate', 'boundary_pct', 'balls_per_boundary', 'pp_balls_bowled', 'pp_runs_conceded', 'pp_wickets', 'pp_dot_balls', 'pp_boundaries', 'pp_innings_bowled', 'pp_over', 'pp_ball', 'pp_overs_bowled', 'pp_economy', 'pp_bowling_average', 'pp_bowling_strike_rate', 'pp_boundary_pct', 'pp_balls_per_boundary', 'middle_balls_bowled', 'middle_runs_conceded', 'middle_wickets', 'middle_dot_balls', 'middle_boundaries', 'middle_innings_bowled', 'middle_over', 'middle_ball', 'middle_overs_bowled', 'middle_economy', 'middle_bowling_average', 'middle_bowling_strike_rate', 'middle_boundary_pct', 'middle_balls_per_boundary', 'death_balls_bowled', 'death_runs_conceded', 'death_wickets', 'death_dot_balls', 'death_boundaries', 'death_innings_bowled', 'death_over', 'deat

,player_name,id,team,balls_bowled,runs_conceded,wickets,dot_balls,boundaries,innings_bowled,over,ball,overs_bowled,economy,bowling_average,bowling_strike_rate,boundary_pct,balls_per_boundary,pp_balls_bowled,pp_runs_conceded,pp_wickets,pp_dot_balls,pp_boundaries,pp_innings_bowled,pp_over,pp_ball,pp_overs_bowled,pp_economy,pp_bowling_average,pp_bowling_strike_rate,pp_boundary_pct,pp_balls_per_boundary,middle_balls_bowled,middle_runs_conceded,middle_wickets,middle_dot_balls,middle_boundaries,middle_innings_bowled,middle_over,middle_ball,middle_overs_bowled,middle_economy,middle_bowling_average,middle_bowling_strike_rate,middle_boundary_pct,middle_balls_per_boundary,death_balls_bowled,death_runs_conceded,death_wickets,death_dot_balls,death_boundaries,death_innings_bowled,death_over,death_ball,death_overs_bowled,death_economy,death_bowling_average,death_bowling_strike_rate,death_boundary_pct,death_balls_per_boundary
0,A Badoni,872b03f7,Lucknow Super Giants,6.0,14.0,0.0,0.0,2.0,1.0,1.0,0.0,1.0,14.00,NaN,NaN,33.33,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,6.0,14.0,0.0,0.0,2.0,1.0,1.0,0.0,1.0,14.00,0.00,0.00,33.33,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0
1,A Kamboj,fcc21ace,Chennai Super Kings,302.0,530.0,21.0,112.0,83.0,14.0,50.0,2.0,50.2,10.53,25.24,14.38,27.48,3.0,126.0,218.0,5.0,53.0,37.0,14.0,21.0,0.0,21.0,10.38,43.60,25.20,29.37,3.0,48.0,77.0,3.0,16.0,11.0,7.0,8.0,0.0,8.0,9.62,25.67,16.00,22.92,4.0,128.0,235.0,13.0,43.0,35.0,12.0,21.0,2.0,21.2,11.02,18.08,9.85,27.34,3.0
2,A Mhatre,b2b4f545,Chennai Super Kings,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A Nortje,acdc62f5,Lucknow Super Giants,24.0,39.0,0.0,7.0,3.0,1.0,4.0,0.0,4.0,9.75,NaN,NaN,12.50,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,18.0,26.0,0.0,4.0,2.0,1.0,3.0,0.0,3.0,8.67,0.00,0.00,11.11,9.0,6.0,13.0,0.0,3.0,1.0,1.0,1.0,0.0,1.0,13.00,0.00,0.00,16.67,6.0
4,A Raghuvanshi,d7017798,Kolkata Knight Riders,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AA Kulkarni,2d140b79,Lucknow Super Giants,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AF Milne,350bb1b1,Rajasthan Royals,20.0,41.0,0.0,2.0,4.0,1.0,3.0,2.0,3.2,12.30,NaN,NaN,20.00,5.0,12.0,22.0,0.0,1.0,2.0,1.0,2.0,0.0,2.0,11.00,0.00,0.00,16.67,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,8.0,19.0,0.0,1.0,2.0,1.0,1.0,2.0,1.2,14.25,0.00,0.00,25.00,4.0
7,AJ Hosein,4d7f517e,Chennai Super Kings,148.0,198.0,8.0,58.0,25.0,7.0,24.0,4.0,24.4,8.03,24.75,18.50,16.89,5.0,66.0,93.0,3.0,37.0,17.0,6.0,11.0,0.0,11.0,8.45,31.00,22.00,25.76,3.0,72.0,91.0,4.0,18.0,7.0,6.0,12.0,0.0,12.0,7.58,22.75,18.00,9.72,10.0,10.0,14.0,1.0,3.0,1.0,2.0,1.0,4.0,1.4,8.40,14.00,10.00,10.00,10.0
8,AK Markram,6a26221c,Lucknow Super Giants,36.0,83.0,0.0,8.0,11.0,5.0,6.0,0.0,6.0,13.83,NaN,NaN,30.56,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.0,24.0,54.0,0.0,6.0,7.0,3.0,4.0,0.0,4.0,13.50,0.00,0.00,29.17,3.0,12.0,29.0,0.0,2.0,4.0,2.0,2.0,0.0,2.0,14.50,0.00,0.00,33.33,3.0
9,AM Ghazanfar,9ca68676,Mumbai Indians,233.0,389.0,15.0,69.0,53.0,11.0,38.0,5.0,38.5,10.02,25.93,15.53,22.75,4.0,54.0,99.0,6.0,22.0,17.0,7.0,9.0,0.0,9.0,11.00,16.50,9.00,31.48,3.0,144.0,216.0,5.0,32.0,23.0,11.0,24.0,0.0,24.0,9.00,43.20,28.80,15.97,6.0,35.0,74.0,4.0,15.0,13.0,6.0,5.0,5.0,5.5,12.69,18.50,8.75,37.14,2.0


In [53]:
player_bowling_stats_df[player_bowling_stats_df["player_name"].str.contains("Archer")]

,player_name,id,team,balls_bowled,runs_conceded,wickets,dot_balls,boundaries,innings_bowled,over,ball,overs_bowled,economy,bowling_average,bowling_strike_rate,boundary_pct,balls_per_boundary,pp_balls_bowled,pp_runs_conceded,pp_wickets,pp_dot_balls,pp_boundaries,pp_innings_bowled,pp_over,pp_ball,pp_overs_bowled,pp_economy,pp_bowling_average,pp_bowling_strike_rate,pp_boundary_pct,pp_balls_per_boundary,middle_balls_bowled,middle_runs_conceded,middle_wickets,middle_dot_balls,middle_boundaries,middle_innings_bowled,middle_over,middle_ball,middle_overs_bowled,middle_economy,middle_bowling_average,middle_bowling_strike_rate,middle_boundary_pct,middle_balls_per_boundary,death_balls_bowled,death_runs_conceded,death_wickets,death_dot_balls,death_boundaries,death_innings_bowled,death_over,death_ball,death_overs_bowled,death_economy,death_bowling_average,death_bowling_strike_rate,death_boundary_pct,death_balls_per_boundary
62,JC Archer,5574750c,Rajasthan Royals,312.0,456.0,21.0,140.0,72.0,14.0,52.0,0.0,52.0,8.77,21.71,14.86,23.08,4.0,174.0,269.0,11.0,90.0,46.0,14.0,29.0,0.0,29.0,9.28,24.45,15.82,26.44,3.0,48.0,56.0,3.0,21.0,9.0,7.0,8.0,0.0,8.0,7.0,18.67,16.0,18.75,5.0,90.0,131.0,7.0,29.0,17.0,11.0,15.0,0.0,15.0,8.73,18.71,12.86,18.89,5.0


In [58]:
player_bowling_stats_df[player_bowling_stats_df["player_name"].str.contains("Rabada")]

,player_name,id,team,balls_bowled,runs_conceded,wickets,dot_balls,boundaries,innings_bowled,over,ball,overs_bowled,economy,bowling_average,bowling_strike_rate,boundary_pct,balls_per_boundary,pp_balls_bowled,pp_runs_conceded,pp_wickets,pp_dot_balls,pp_boundaries,pp_innings_bowled,pp_over,pp_ball,pp_overs_bowled,pp_economy,pp_bowling_average,pp_bowling_strike_rate,pp_boundary_pct,pp_balls_per_boundary,middle_balls_bowled,middle_runs_conceded,middle_wickets,middle_dot_balls,middle_boundaries,middle_innings_bowled,middle_over,middle_ball,middle_overs_bowled,middle_economy,middle_bowling_average,middle_bowling_strike_rate,middle_boundary_pct,middle_balls_per_boundary,death_balls_bowled,death_runs_conceded,death_wickets,death_dot_balls,death_boundaries,death_innings_bowled,death_over,death_ball,death_overs_bowled,death_economy,death_bowling_average,death_bowling_strike_rate,death_boundary_pct,death_balls_per_boundary
71,K Rabada,e62dd25d,Gujarat Titans,322.0,493.0,24.0,142.0,76.0,14.0,53.0,4.0,53.4,9.19,20.54,13.42,23.6,4.0,222.0,342.0,17.0,102.0,55.0,14.0,37.0,0.0,37.0,9.24,20.12,13.06,24.77,4.0,40.0,59.0,3.0,18.0,10.0,7.0,6.0,4.0,6.4,8.85,19.67,13.33,25.0,4.0,60.0,92.0,4.0,22.0,11.0,10.0,10.0,0.0,10.0,9.2,23.0,15.0,18.33,5.0


In [55]:
player_bowling_stats_df[player_bowling_stats_df["player_name"].str.contains("Holder")]

,player_name,id,team,balls_bowled,runs_conceded,wickets,dot_balls,boundaries,innings_bowled,over,ball,overs_bowled,economy,bowling_average,bowling_strike_rate,boundary_pct,balls_per_boundary,pp_balls_bowled,pp_runs_conceded,pp_wickets,pp_dot_balls,pp_boundaries,pp_innings_bowled,pp_over,pp_ball,pp_overs_bowled,pp_economy,pp_bowling_average,pp_bowling_strike_rate,pp_boundary_pct,pp_balls_per_boundary,middle_balls_bowled,middle_runs_conceded,middle_wickets,middle_dot_balls,middle_boundaries,middle_innings_bowled,middle_over,middle_ball,middle_overs_bowled,middle_economy,middle_bowling_average,middle_bowling_strike_rate,middle_boundary_pct,middle_balls_per_boundary,death_balls_bowled,death_runs_conceded,death_wickets,death_dot_balls,death_boundaries,death_innings_bowled,death_over,death_ball,death_overs_bowled,death_economy,death_bowling_average,death_bowling_strike_rate,death_boundary_pct,death_balls_per_boundary
68,JO Holder,0f721006,Gujarat Titans,170.0,208.0,13.0,68.0,26.0,8.0,28.0,2.0,28.2,7.34,16.0,13.08,15.29,6.0,6.0,8.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,8.0,0.0,0.0,16.67,6.0,138.0,175.0,9.0,57.0,22.0,8.0,23.0,0.0,23.0,7.61,19.44,15.33,15.94,6.0,26.0,25.0,4.0,10.0,3.0,5.0,4.0,2.0,4.2,5.77,6.25,6.5,11.54,8.0


In [56]:
player_bowling_stats_df[player_bowling_stats_df["player_name"].str.contains("Bumrah")]

,player_name,id,team,balls_bowled,runs_conceded,wickets,dot_balls,boundaries,innings_bowled,over,ball,overs_bowled,economy,bowling_average,bowling_strike_rate,boundary_pct,balls_per_boundary,pp_balls_bowled,pp_runs_conceded,pp_wickets,pp_dot_balls,pp_boundaries,pp_innings_bowled,pp_over,pp_ball,pp_overs_bowled,pp_economy,pp_bowling_average,pp_bowling_strike_rate,pp_boundary_pct,pp_balls_per_boundary,middle_balls_bowled,middle_runs_conceded,middle_wickets,middle_dot_balls,middle_boundaries,middle_innings_bowled,middle_over,middle_ball,middle_overs_bowled,middle_economy,middle_bowling_average,middle_bowling_strike_rate,middle_boundary_pct,middle_balls_per_boundary,death_balls_bowled,death_runs_conceded,death_wickets,death_dot_balls,death_boundaries,death_innings_bowled,death_over,death_ball,death_overs_bowled,death_economy,death_bowling_average,death_bowling_strike_rate,death_boundary_pct,death_balls_per_boundary
66,JJ Bumrah,462411b3,Mumbai Indians,294.0,410.0,4.0,105.0,55.0,13.0,49.0,0.0,49.0,8.37,102.5,73.5,18.71,5.0,144.0,211.0,2.0,57.0,32.0,13.0,24.0,0.0,24.0,8.79,105.5,72.0,22.22,4.0,72.0,93.0,1.0,26.0,10.0,10.0,12.0,0.0,12.0,7.75,93.0,72.0,13.89,7.0,78.0,106.0,1.0,22.0,13.0,9.0,13.0,0.0,13.0,8.15,106.0,78.0,16.67,6.0


In [57]:
player_bowling_stats_df[player_bowling_stats_df["player_name"].str.contains("Narine")]

,player_name,id,team,balls_bowled,runs_conceded,wickets,dot_balls,boundaries,innings_bowled,over,ball,overs_bowled,economy,bowling_average,bowling_strike_rate,boundary_pct,balls_per_boundary,pp_balls_bowled,pp_runs_conceded,pp_wickets,pp_dot_balls,pp_boundaries,pp_innings_bowled,pp_over,pp_ball,pp_overs_bowled,pp_economy,pp_bowling_average,pp_bowling_strike_rate,pp_boundary_pct,pp_balls_per_boundary,middle_balls_bowled,middle_runs_conceded,middle_wickets,middle_dot_balls,middle_boundaries,middle_innings_bowled,middle_over,middle_ball,middle_overs_bowled,middle_economy,middle_bowling_average,middle_bowling_strike_rate,middle_boundary_pct,middle_balls_per_boundary,death_balls_bowled,death_runs_conceded,death_wickets,death_dot_balls,death_boundaries,death_innings_bowled,death_over,death_ball,death_overs_bowled,death_economy,death_bowling_average,death_bowling_strike_rate,death_boundary_pct,death_balls_per_boundary
161,SP Narine,9d430b40,Kolkata Knight Riders,306.0,339.0,15.0,131.0,40.0,13.0,51.0,0.0,51.0,6.65,22.6,20.4,13.07,7.0,72.0,96.0,2.0,33.0,16.0,10.0,12.0,0.0,12.0,8.0,48.0,36.0,22.22,4.0,180.0,194.0,4.0,68.0,18.0,13.0,30.0,0.0,30.0,6.47,48.5,45.0,10.0,10.0,54.0,49.0,9.0,30.0,6.0,9.0,9.0,0.0,9.0,5.44,5.44,6.0,11.11,9.0


In [ ]:
def calculate_player_stats(deliveries_df, players_df):
    """
    Calculates overall season stats for every player from the deliveries DataFrame.
    Returns a single player_stats_df with one row per player containing:
      - Batting stats
      - Bowling stats
      - Fielding stats
    All joined to the players table on player_name.

    Corrections applied:
      - Wides excluded from balls faced (no-balls count as faced)
      - Runs conceded includes wides and no-balls (but not byes or leg byes)
      - Dismissals counted via player_dismissed (catches non-striker run outs too)
    """
    


    # -------------------------------------------------------
    # BOWLING STATS
    # grouped by bowler
    # -------------------------------------------------------

    # A legal delivery excludes wides and no-balls
    deliveries_df["is_legal"] = (
        (deliveries_df["extra_type"] != "wides") & (deliveries_df["extra_type"] != "noballs")
    ).astype(int)

    # Runs conceded by bowler = runs off bat + wides + no-balls
    # Byes and leg byes are NOT the bowler's fault
    deliveries_df["runs_conceded"] = (
        deliveries_df["runs_off_bat"].fillna(0) +
        deliveries_df["wides"].fillna(0) +
        deliveries_df["noballs"].fillna(0)
    )

    # Wickets credited to bowler — excludes run outs, retired hurt, obstructions
    deliveries_df["bowler_wicket"] = (
        deliveries_df["wicket_type"].notna() &
        ~deliveries_df["wicket_type"].isin([
            "run out", "retired hurt", "retired out", "obstructing the field"
        ])
    ).astype(int)

    # Dot balls = legal delivery where total runs == 0
    deliveries_df["total_runs"] = deliveries_df["runs_off_bat"] + deliveries_df["extras"]
    deliveries_df["is_dot"] = (
        (deliveries_df["is_legal"] == 1) & (deliveries_df["total_runs"] == 0)
    ).astype(int)

    bowling_stats = deliveries_df.groupby("bowler").agg(
        balls_bowled   = ("is_legal",        "sum"),
        runs_conceded  = ("runs_conceded",    "sum"),
        wickets        = ("bowler_wicket",    "sum"),
        dot_balls      = ("is_dot",           "sum"),
        innings_bowled = ("match_id",         "nunique")
    ).reset_index().rename(columns={"bowler": "player_name"})

    # Overs bowled (balls / 6, rounded to 1 decimal)
    bowling_stats["overs_bowled"] = (
        bowling_stats["balls_bowled"] / 6
    ).round(1)

    # Economy rate = runs conceded per over
    bowling_stats["economy"] = (
        bowling_stats["runs_conceded"] / bowling_stats["overs_bowled"]
    ).round(2)

    # Bowling average = runs conceded per wicket
    bowling_stats["bowling_average"] = (
        bowling_stats["runs_conceded"] / bowling_stats["wickets"].replace(0, float("nan"))
    ).round(2)

    # Bowling strike rate = balls bowled per wicket
    bowling_stats["bowling_strike_rate"] = (
    bowling_stats["balls_bowled"] / bowling_stats["wickets"].replace(0, float("nan"))
    ).round(2)


    # -------------------------------------------------------
    # FIELDING STATS
    # -------------------------------------------------------

    # Caught and bowled — catch credited to the bowler
    catches_cb = deliveries_df[
        deliveries_df["wicket_type"] == "caught and bowled"
    ].groupby("bowler").size().reset_index()
    catches_cb.columns = ["player_name", "catches_cb"]

    # Normal catches — fielder is in other_player_dismissed
    catches_normal = deliveries_df[
        deliveries_df["wicket_type"] == "caught"
    ].groupby("other_player_dismissed").size().reset_index()
    catches_normal.columns = ["player_name", "catches_normal"]

    # Run outs — credited to fielder in other_player_dismissed
    run_outs = deliveries_df[
        deliveries_df["wicket_type"] == "run out"
    ].groupby("other_player_dismissed").size().reset_index()
    run_outs.columns = ["player_name", "run_outs"]

    # Stumpings — credited to wicketkeeper in other_player_dismissed
    stumpings = deliveries_df[
        deliveries_df["wicket_type"] == "stumped"
    ].groupby("other_player_dismissed").size().reset_index()
    stumpings.columns = ["player_name", "stumpings"]

    # Combine all fielding stats
    fielding_stats = (catches_normal
        .merge(catches_cb,  on="player_name", how="outer")
        .merge(run_outs,    on="player_name", how="outer")
        .merge(stumpings,   on="player_name", how="outer")
        .fillna(0)
    )
    fielding_stats["catches"] = (
        fielding_stats["catches_normal"] + fielding_stats["catches_cb"]
    ).astype(int)
    fielding_stats = fielding_stats[["player_name", "catches", "run_outs", "stumpings"]]


    # -------------------------------------------------------
    # JOIN EVERYTHING TOGETHER
    # -------------------------------------------------------
    player_stats_df = (players_df
        .merge(batting_stats,  on="player_name", how="left")
        .merge(bowling_stats,  on="player_name", how="left")
        .merge(fielding_stats, on="player_name", how="left")
        .fillna(0)
    )

    print(f"✓ Player stats calculated: {len(player_stats_df)} players")
    print(f"  Columns: {list(player_stats_df.columns)}")

    return player_stats_df

In [ ]:
player_stats_df = calculate_player_stats(deliveries_df, players_df)


In [ ]:
def calculate_innings_stats(deliveries_df):
    """
    Calculates per-innings stats for each player.
    Used to derive:
      Batting: highest score (with not out *), 50s, 100s
      Bowling: best bowling figures, 4 wicket hauls, 5 wicket hauls

    Returns:
      batting_innings_df : one row per batter per match per innings
      bowling_innings_df : one row per bowler per match per innings
    """

    # -------------------------------------------------------
    # BATTING INNINGS STATS
    # group by match + innings + striker
    # -------------------------------------------------------

    # Reuse ball_faced and is_four, is_six if already calculated
    # otherwise calculate them here too
    if "ball_faced" not in deliveries_df.columns:
        deliveries_df["ball_faced"] = deliveries_df["wides"].isna().astype(int)
    if "is_four" not in deliveries_df.columns:
        deliveries_df["is_four"] = (deliveries_df["runs_off_bat"] == 4).astype(int)
    if "is_six" not in deliveries_df.columns:
        deliveries_df["is_six"] = (deliveries_df["runs_off_bat"] == 6).astype(int)

    batting_innings_df = deliveries_df.groupby(
        ["match_id", "innings", "striker"]
    ).agg(
        runs        = ("runs_off_bat", "sum"),
        balls_faced = ("ball_faced",   "sum"),
        fours       = ("is_four",      "sum"),
        sixes       = ("is_six",       "sum"),
    ).reset_index().rename(columns={"striker": "player_name"})

    # Determine if batter was dismissed in this innings
    # by checking if their name appears in player_dismissed
    # for the same match and innings
    dismissals = (deliveries_df[deliveries_df["player_dismissed"].notna()]
                    [["match_id", "innings", "player_dismissed"]]
                    .drop_duplicates()
                    .rename(columns={"player_dismissed": "player_name"}))
    dismissals["was_dismissed"] = True

    batting_innings_df = batting_innings_df.merge(
        dismissals,
        on=["match_id", "innings", "player_name"],
        how="left"
    )
    batting_innings_df["was_dismissed"] = batting_innings_df["was_dismissed"].fillna(False)

    # Format highest score with * for not out
    batting_innings_df["score_str"] = batting_innings_df.apply(
        lambda row: str(int(row["runs"])) if row["was_dismissed"] else str(int(row["runs"])) + "*",
        axis=1
    )

    # Flag 50s and 100s
    # 50 = runs >= 50 and < 100
    # 100 = runs >= 100
    batting_innings_df["is_fifty"]   = (
        (batting_innings_df["runs"] >= 50) & (batting_innings_df["runs"] < 100)
    ).astype(int)
    batting_innings_df["is_hundred"] = (
        batting_innings_df["runs"] >= 100
    ).astype(int)


    # -------------------------------------------------------
    # BOWLING INNINGS STATS
    # group by match + innings + bowler
    # -------------------------------------------------------

    if "is_legal" not in deliveries_df.columns:
        deliveries_df["is_legal"] = (
            deliveries_df["wides"].isna() & deliveries_df["noballs"].isna()
        ).astype(int)

    if "runs_conceded" not in deliveries_df.columns:
        deliveries_df["runs_conceded"] = (
            deliveries_df["runs_off_bat"].fillna(0) +
            deliveries_df["wides"].fillna(0) +
            deliveries_df["noballs"].fillna(0)
        )

    if "bowler_wicket" not in deliveries_df.columns:
        deliveries_df["bowler_wicket"] = (
            deliveries_df["wicket_type"].notna() &
            ~deliveries_df["wicket_type"].isin([
                "run out", "retired hurt", "retired out", "obstructing the field"
            ])
        ).astype(int)

    bowling_innings_df = deliveries_df.groupby(
        ["match_id", "innings", "bowler"]
    ).agg(
        balls_bowled  = ("is_legal",       "sum"),
        runs_conceded = ("runs_conceded",  "sum"),
        wickets       = ("bowler_wicket",  "sum"),
    ).reset_index().rename(columns={"bowler": "player_name"})

    # Flag 4 wicket and 5 wicket hauls
    bowling_innings_df["is_four_wickets"] = (
        (bowling_innings_df["wickets"] >= 4) & (bowling_innings_df["wickets"] < 5)
    ).astype(int)
    bowling_innings_df["is_five_wickets"] = (
        bowling_innings_df["wickets"] >= 5
    ).astype(int)

    # Best bowling figures as string e.g. "4/22"
    bowling_innings_df["figures_str"] = (
        bowling_innings_df["wickets"].astype(int).astype(str) + "/" +
        bowling_innings_df["runs_conceded"].astype(int).astype(str)
    )

    return batting_innings_df, bowling_innings_df


def calculate_milestones(batting_innings_df, bowling_innings_df):
    """
    Aggregates innings-level data into per-player milestone stats:
      Batting : highest_score, fifties, hundreds
      Bowling : best_figures, four_wicket_hauls, five_wicket_hauls
    """

    # -------------------------------------------------------
    # BATTING MILESTONES
    # -------------------------------------------------------

    # Highest score — sort by runs desc, not out ranks above out for equal runs
    batting_innings_df["not_out"] = (~batting_innings_df["was_dismissed"]).astype(int)

    highest_score = (batting_innings_df
        .sort_values(["runs", "not_out"], ascending=[False, False])
        .groupby("player_name")
        .first()[["score_str"]]
        .reset_index()
        .rename(columns={"score_str": "highest_score"})
    )

    # Count 50s and 100s per player
    batting_milestones = batting_innings_df.groupby("player_name").agg(
        fifties  = ("is_fifty",   "sum"),
        hundreds = ("is_hundred", "sum"),
    ).reset_index()

    batting_milestones = batting_milestones.merge(highest_score, on="player_name", how="left")


    # -------------------------------------------------------
    # BOWLING MILESTONES
    # -------------------------------------------------------

    # Best bowling figures — sort by wickets desc, then runs asc (fewer runs is better)
    best_figures = (bowling_innings_df
        .sort_values(["wickets", "runs_conceded"], ascending=[False, True])
        .groupby("player_name")
        .first()[["figures_str"]]
        .reset_index()
        .rename(columns={"figures_str": "best_figures"})
    )

    # Count 4 and 5 wicket hauls per player
    bowling_milestones = bowling_innings_df.groupby("player_name").agg(
        four_wicket_hauls = ("is_four_wickets", "sum"),
        five_wicket_hauls = ("is_five_wickets", "sum"),
    ).reset_index()

    bowling_milestones = bowling_milestones.merge(best_figures, on="player_name", how="left")

    return batting_milestones, bowling_milestones

In [ ]:
batting_innings_df, bowling_innings_df = calculate_innings_stats(deliveries_df)

In [ ]:
batting_milestones, bowling_milestones = calculate_milestones(batting_innings_df, bowling_innings_df)

In [ ]:
player_stats_df = player_stats_df.merge(batting_milestones, on="player_name", how='left')
player_stats_df = player_stats_df.merge(bowling_milestones, on="player_name", how='left')

In [ ]:
player_stats_df

In [ ]:
matches_df[matches_df['match_number']=='50']